# FPT AI VLM Captioning cho Selected Shots — Adaptive + Multi-worker

Pipeline:

`selected_shots.csv + videos.txt → download video → adaptive segment → contact sheet → FPT AI VLM → checkpoint từng segment → checkpoint từng shot → captions.sql`

## Shot dài được xử lý thế nào?

Không cố nhét quá nhiều frame vào một ảnh.

Mặc định:

- `<= 15s` → 1 segment → 1 contact sheet × 6 frame → 1 API call
- `15–30s` → 2 segment → 2 API call
- `30–45s` → 3 segment → 3 API call
- `> 45s` → tối đa 4 segment → 4 API call

Với shot rất dài, toàn bộ timeline vẫn được chia đều thành tối đa 4 đoạn để giữ coverage từ đầu đến cuối.

FPT AI chỉ nhận 1 ảnh/request, vì vậy mỗi segment được gọi riêng. Nếu shot có nhiều segment, mỗi segment được yêu cầu mô tả bằng **1 câu ngắn**, sau đó các câu được ghép theo thứ tự thời gian thành caption cuối.

## Khả năng resume

- Mỗi **segment thành công** được ghi ngay vào `segments_checkpoint.jsonl`.
- Khi đủ segment của một shot, caption cuối được ghi ngay vào `shots_checkpoint.jsonl`.
- `captions.sql` được flush định kỳ.
- Nếu runtime dừng giữa chừng, chạy lại Notebook sẽ bỏ qua các segment/shot đã hoàn thành.

In [ ]:
# ============================================================
# CELL 1 - CONFIG
# ============================================================

# ---------------------------
# FPT AI
# ---------------------------
FPT_API_KEY = ""
FPT_BASE_URL = "https://mkp-api.fptcloud.com"
MODEL_NAME = "YOUR_VLM_MODEL_NAME"

# ---------------------------
# INPUT
# ---------------------------
# Để None -> tự tìm trong /kaggle/input
SELECTED_SHOTS_CSV = None
VIDEOS_TXT = None
INPUT_ROOT = "/kaggle/input"

# Có thể shard selected_shots.csv theo thứ tự dòng.
# Header CSV không được tính là shot.
# SHOT_END là inclusive.
SHOT_START = 0
SHOT_END = None  # ví dụ 4999; None = chạy tới cuối file

# ---------------------------
# OUTPUT
# ---------------------------
OUTPUT_DIR = "/kaggle/working/caption_output"
CAPTION_TABLE = "caption"

# ---------------------------
# WORKERS
# ---------------------------
DOWNLOAD_WORKERS = 2
PROCESS_WORKERS = 6
API_WORKERS = 8

# Số video giữ trên disk cùng lúc
VIDEO_BATCH_SIZE = 2

# Mỗi lần chuẩn bị tối đa bao nhiêu shot.
# Giữ nhỏ để không tạo quá nhiều contact sheet trên disk.
SHOT_CHUNK_SIZE = 40

# ---------------------------
# ADAPTIVE SHOT SAMPLING
# ---------------------------
TARGET_SECONDS_PER_SEGMENT = 15.0
FRAMES_PER_SEGMENT = 6
MAX_SEGMENTS_PER_SHOT = 4

# Contact sheet
MONTAGE_COLS = 3
FRAME_WIDTH = 400
FRAME_HEIGHT = 225
JPEG_QUALITY = 88

# ---------------------------
# GENERATION
# ---------------------------
MAX_TOKENS = 300
TEMPERATURE = 0.0

# Flush lại captions.sql sau mỗi N shot hoàn thành.
# Segment/shot checkpoint vẫn ghi NGAY sau mỗi kết quả.
SQL_FLUSH_EVERY = 10

In [ ]:
# ============================================================
# CELL 2 - INSTALL
# ============================================================
# Không upgrade Pillow/Pandas/OpenCV để tránh xung đột Kaggle runtime.

!pip -q install -U openai

In [ ]:
# ============================================================
# CELL 3 - IMPORTS + AUTO-DETECT INPUT
# ============================================================

import os
import re
import cv2
import json
import time
import math
import base64
import requests
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image, ImageOps
from openai import OpenAI
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


def find_unique_file(explicit_path, candidate_names):
    if explicit_path is not None:
        p = Path(explicit_path)
        if not p.exists():
            raise FileNotFoundError(p)
        return str(p)

    matches = []
    root = Path(INPUT_ROOT)

    for name in candidate_names:
        matches.extend(root.rglob(name))

    matches = list(dict.fromkeys(matches))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Không tìm thấy {candidate_names} bên dưới {INPUT_ROOT}"
        )

    if len(matches) > 1:
        print("Tìm thấy nhiều file phù hợp:")
        for p in matches:
            print(" -", p)

        raise RuntimeError(
            "Hãy set path cụ thể trong CELL 1 để tránh chọn nhầm file."
        )

    return str(matches[0])


SELECTED_SHOTS_CSV = find_unique_file(
    SELECTED_SHOTS_CSV,
    ["selected_shots.csv", "shots_selected.csv"],
)

VIDEOS_TXT = find_unique_file(
    VIDEOS_TXT,
    ["videos.txt", "video.txt"],
)

OUTPUT_DIR = Path(OUTPUT_DIR)
VIDEO_DIR = OUTPUT_DIR / "videos"
MONTAGE_DIR = OUTPUT_DIR / "montages"

SEGMENT_CHECKPOINT_FILE = OUTPUT_DIR / "segments_checkpoint.jsonl"
SHOT_CHECKPOINT_FILE = OUTPUT_DIR / "shots_checkpoint.jsonl"

SQL_FILE = OUTPUT_DIR / "captions.sql"
FAILED_FILE = OUTPUT_DIR / "failed.csv"

for folder in [OUTPUT_DIR, VIDEO_DIR, MONTAGE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("SELECTED_SHOTS_CSV:", SELECTED_SHOTS_CSV)
print("VIDEOS_TXT:", VIDEOS_TXT)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# ============================================================
# CELL 4 - LOAD + VALIDATE DATA
# ============================================================

def load_videos_txt(path):
    df = pd.read_csv(path, skipinitialspace=True)

    if not {"video_id", "video_url"}.issubset(df.columns):
        df = pd.read_csv(
            path,
            names=["video_id", "video_url"],
            header=None,
            skipinitialspace=True,
        )

    df["video_id"] = df["video_id"].astype(str).str.strip()
    df["video_url"] = df["video_url"].astype(str).str.strip()

    return df[["video_id", "video_url"]]


videos_df = load_videos_txt(VIDEOS_TXT)

# Dòng đầu mặc định là header -> Pandas không tính vào dữ liệu.
shots_df = pd.read_csv(SELECTED_SHOTS_CSV)

required_cols = {
    "shot_id",
    "video_id",
    "shot_index",
    "start_ms",
    "end_ms",
}

missing = required_cols - set(shots_df.columns)

if missing:
    raise ValueError(
        f"selected_shots.csv thiếu các cột: {sorted(missing)}"
    )

shots_df["shot_id"] = shots_df["shot_id"].astype(str).str.strip()
shots_df["video_id"] = shots_df["video_id"].astype(str).str.strip()

if shots_df["shot_id"].duplicated().any():
    duplicated = shots_df[
        shots_df["shot_id"].duplicated(keep=False)
    ]

    raise ValueError(
        "shot_id bị trùng:\n"
        + duplicated[
            ["shot_id", "video_id", "shot_index"]
        ].head(20).to_string(index=False)
    )

bad_time = shots_df[
    shots_df["end_ms"] <= shots_df["start_ms"]
]

if not bad_time.empty:
    raise ValueError(
        "Có shot end_ms <= start_ms:\n"
        + bad_time[
            ["shot_id", "video_id", "start_ms", "end_ms"]
        ].head(20).to_string(index=False)
    )

# Shard theo thứ tự dòng.
if SHOT_START < 0:
    raise ValueError("SHOT_START phải >= 0")

if SHOT_START >= len(shots_df):
    raise ValueError(
        f"SHOT_START={SHOT_START} vượt quá số shot ({len(shots_df)})."
    )

if SHOT_END is None:
    end_idx = len(shots_df) - 1
else:
    if SHOT_END < SHOT_START:
        raise ValueError("SHOT_END phải >= SHOT_START")

    # Nếu tràn file -> tự co về shot cuối.
    end_idx = min(SHOT_END, len(shots_df) - 1)

selected_df = shots_df.iloc[
    SHOT_START:end_idx + 1
].copy()

selected_df["_source_row"] = selected_df.index.astype(int)

video_url_map = dict(
    zip(videos_df["video_id"], videos_df["video_url"])
)

missing_video_ids = sorted(
    set(selected_df["video_id"]) - set(video_url_map)
)

if missing_video_ids:
    raise ValueError(
        "videos.txt thiếu URL cho: "
        + ", ".join(missing_video_ids[:20])
    )

print(f"Tổng selected shot trong file: {len(shots_df):,}")
print(
    f"Range đang chạy: {SHOT_START} -> {end_idx} "
    f"(inclusive)"
)
print(f"Số shot chạy: {len(selected_df):,}")
print(
    f"Số video liên quan: "
    f"{selected_df['video_id'].nunique():,}"
)

display(
    selected_df[
        ["shot_id", "video_id", "shot_index", "start_ms", "end_ms"]
    ].head(10)
)

In [ ]:
# ============================================================
# CELL 5 - PROMPT
# ============================================================

SYSTEM_PROMPT = """
Bạn là hệ thống tạo caption cho shot video phục vụ bài toán video retrieval.

Ảnh đầu vào là một contact sheet gồm nhiều frame thuộc MỘT đoạn thời gian
của một shot. Các frame được sắp xếp theo thời gian từ trái sang phải,
từ trên xuống dưới.

Yêu cầu:
- Mô tả sự kiện/hành động chính.
- Mô tả nhân vật/chủ thể chính; nếu thấy rõ thì nêu số lượng tương đối,
  trang phục, màu sắc hoặc đặc điểm thị giác nổi bật.
- Mô tả bối cảnh, không gian và vật thể quan trọng.
- Ưu tiên chi tiết hữu ích cho việc tìm kiếm lại video.

Quy tắc:
- KHÔNG OCR; không chép phụ đề, biển hiệu, logo hay chữ trong hình.
- Không suy đoán danh tính, tên riêng hoặc địa điểm cụ thể khi không chắc chắn.
- Không nhắc tới contact sheet, ảnh ghép, frame hay cách ảnh được tạo.
- Chỉ trả về caption, không tiêu đề, không bullet, không giải thích thêm.
""".strip()


def build_user_prompt(segment_index, total_segments):
    if total_segments == 1:
        return (
            "Đây là toàn bộ shot. Hãy mô tả shot bằng 2-4 câu tiếng Việt, "
            "nêu rõ hành động chính, chủ thể và bối cảnh."
        )

    return (
        f"Đây là đoạn {segment_index + 1}/{total_segments} theo thứ tự thời gian "
        "của một shot dài. Hãy mô tả đoạn này bằng đúng 1 câu tiếng Việt "
        "ngắn gọn nhưng đủ thông tin quan trọng. Không nhắc số thứ tự đoạn."
    )

In [ ]:
# ============================================================
# CELL 6 - HELPER FUNCTIONS
# ============================================================

client = OpenAI(
    api_key=FPT_API_KEY,
    base_url=FPT_BASE_URL,
    timeout=120.0,
    max_retries=5,
)


def safe_filename(value):
    return re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(value),
    )


def remove_file(path):
    try:
        Path(path).unlink(missing_ok=True)
    except Exception:
        pass


def download_video(video_id, url, retries=3):
    out_path = (
        VIDEO_DIR
        / f"{safe_filename(video_id)}.mp4"
    )

    part_path = out_path.with_suffix(
        ".mp4.part"
    )

    if (
        out_path.exists()
        and out_path.stat().st_size > 0
    ):
        return video_id, out_path

    for attempt in range(1, retries + 1):
        try:
            remove_file(part_path)

            with requests.get(
                url,
                stream=True,
                timeout=(20, 300),
                headers={
                    "User-Agent": "Mozilla/5.0"
                },
            ) as r:
                r.raise_for_status()

                with open(part_path, "wb") as f:
                    for chunk in r.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            f.write(chunk)

            part_path.replace(out_path)

            return video_id, out_path

        except Exception:
            remove_file(part_path)

            if attempt == retries:
                raise

            time.sleep(2 ** attempt)


def num_segments_for_shot(start_ms, end_ms):
    duration_sec = max(
        0.001,
        (float(end_ms) - float(start_ms))
        / 1000.0,
    )

    n = math.ceil(
        duration_sec
        / TARGET_SECONDS_PER_SEGMENT
    )

    return max(
        1,
        min(MAX_SEGMENTS_PER_SHOT, n),
    )


def get_segment_ranges(start_ms, end_ms):
    n = num_segments_for_shot(
        start_ms,
        end_ms,
    )

    edges = np.linspace(
        float(start_ms),
        float(end_ms),
        n + 1,
    )

    return [
        (
            int(round(edges[i])),
            int(round(edges[i + 1])),
        )
        for i in range(n)
    ]


def sample_timestamps(
    segment_start_ms,
    segment_end_ms,
    n,
):
    start = float(segment_start_ms)
    end = float(segment_end_ms)

    duration = end - start

    if duration <= 0:
        return [start]

    if n <= 1 or duration < 300:
        return [
            start + duration / 2
        ]

    margin = min(
        duration * 0.05,
        200.0,
    )

    left = start + margin
    right = end - margin

    if right <= left:
        return [
            start + duration / 2
        ]

    return np.linspace(
        left,
        right,
        n,
    ).tolist()


def make_missing_segment_sheets(
    row,
    video_path,
    completed_segment_keys,
):
    """
    Mở video 1 lần cho 1 shot rồi tạo các sheet còn thiếu.
    Return list[dict].
    """
    shot_id = str(row["shot_id"])

    segment_ranges = get_segment_ranges(
        row["start_ms"],
        row["end_ms"],
    )

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise RuntimeError(
            f"Không mở được video: {video_path}"
        )

    results = []

    try:
        for segment_index, (
            seg_start,
            seg_end,
        ) in enumerate(segment_ranges):

            segment_key = (
                shot_id,
                segment_index,
            )

            if (
                segment_key
                in completed_segment_keys
            ):
                continue

            timestamps = sample_timestamps(
                seg_start,
                seg_end,
                FRAMES_PER_SEGMENT,
            )

            frames = []

            for ts in timestamps:
                cap.set(
                    cv2.CAP_PROP_POS_MSEC,
                    float(ts),
                )

                ok, frame = cap.read()

                if not ok or frame is None:
                    continue

                frame = cv2.cvtColor(
                    frame,
                    cv2.COLOR_BGR2RGB,
                )

                frames.append(
                    Image.fromarray(frame)
                )

            if not frames:
                raise RuntimeError(
                    f"Không lấy được frame: "
                    f"{shot_id}, segment "
                    f"{segment_index + 1}/"
                    f"{len(segment_ranges)}"
                )

            cols = min(
                MONTAGE_COLS,
                len(frames),
            )

            rows = math.ceil(
                len(frames) / cols
            )

            sheet = Image.new(
                "RGB",
                (
                    cols * FRAME_WIDTH,
                    rows * FRAME_HEIGHT,
                ),
                "black",
            )

            for i, img in enumerate(frames):
                tile = ImageOps.fit(
                    img,
                    (
                        FRAME_WIDTH,
                        FRAME_HEIGHT,
                    ),
                    method=Image.Resampling.LANCZOS,
                )

                x = (
                    i % cols
                ) * FRAME_WIDTH

                y = (
                    i // cols
                ) * FRAME_HEIGHT

                sheet.paste(
                    tile,
                    (x, y),
                )

            out_path = (
                MONTAGE_DIR
                / (
                    f"{safe_filename(shot_id)}"
                    f"_seg{segment_index:02d}.jpg"
                )
            )

            sheet.save(
                out_path,
                "JPEG",
                quality=JPEG_QUALITY,
                optimize=True,
            )

            results.append({
                "shot_id": shot_id,
                "video_id": str(
                    row["video_id"]
                ),
                "shot_index": int(
                    row["shot_index"]
                ),
                "source_row": int(
                    row["_source_row"]
                ),
                "start_ms": int(
                    row["start_ms"]
                ),
                "end_ms": int(
                    row["end_ms"]
                ),
                "segment_index": (
                    segment_index
                ),
                "total_segments": len(
                    segment_ranges
                ),
                "segment_start_ms": (
                    seg_start
                ),
                "segment_end_ms": (
                    seg_end
                ),
                "image_path": out_path,
            })

    finally:
        cap.release()

    return results


def image_to_data_url(image_path):
    with open(
        image_path,
        "rb",
    ) as f:
        encoded = base64.b64encode(
            f.read()
        ).decode("utf-8")

    return (
        "data:image/jpeg;base64,"
        + encoded
    )


def caption_segment(item):
    user_prompt = build_user_prompt(
        item["segment_index"],
        item["total_segments"],
    )

    response = (
        client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT,
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": image_to_data_url(
                                    item["image_path"]
                                )
                            },
                        },
                        {
                            "type": "text",
                            "text": user_prompt,
                        },
                    ],
                },
            ],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            stream=False,
        )
    )

    text = (
        response.choices[0]
        .message.content
    )

    if not text or not str(text).strip():
        raise RuntimeError(
            "FPT AI trả về caption rỗng."
        )

    return str(text).strip()


def load_segment_checkpoint():
    records = {}

    if not SEGMENT_CHECKPOINT_FILE.exists():
        return records

    with open(
        SEGMENT_CHECKPOINT_FILE,
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            try:
                rec = json.loads(line)

                key = (
                    str(rec["shot_id"]),
                    int(rec["segment_index"]),
                )

                records[key] = rec

            except Exception:
                pass

    return records


def load_shot_checkpoint():
    records = {}

    if not SHOT_CHECKPOINT_FILE.exists():
        return records

    with open(
        SHOT_CHECKPOINT_FILE,
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            try:
                rec = json.loads(line)
                records[
                    str(rec["shot_id"])
                ] = rec
            except Exception:
                pass

    return records


def append_jsonl(path, record):
    """
    Ghi ngay xuống disk.
    """
    with open(
        path,
        "a",
        encoding="utf-8",
    ) as f:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

        f.flush()
        os.fsync(f.fileno())


def build_final_shot_record(
    row,
    segment_records,
):
    shot_id = str(row["shot_id"])

    total_segments = num_segments_for_shot(
        row["start_ms"],
        row["end_ms"],
    )

    segments = []

    for segment_index in range(
        total_segments
    ):
        key = (
            shot_id,
            segment_index,
        )

        if key not in segment_records:
            return None

        segments.append(
            segment_records[key]
        )

    # 1 segment: giữ nguyên caption đầy đủ.
    if total_segments == 1:
        final_caption = segments[0][
            "segment_caption"
        ].strip()

    else:
        # Mỗi segment đã được yêu cầu trả đúng 1 câu.
        final_caption = " ".join(
            seg["segment_caption"].strip()
            for seg in segments
        )

    return {
        "caption_id": (
            f"{shot_id}_0001"
        ),
        "shot_id": shot_id,
        "caption_text": final_caption,
        "video_id": str(
            row["video_id"]
        ),
        "shot_index": int(
            row["shot_index"]
        ),
        "start_ms": int(
            row["start_ms"]
        ),
        "end_ms": int(
            row["end_ms"]
        ),
        "source_row": int(
            row["_source_row"]
        ),
        "num_segments": (
            total_segments
        ),
    }


def sql_literal(value):
    return (
        "'"
        + str(value).replace(
            "'",
            "''",
        )
        + "'"
    )


def write_sql(shot_records):
    ordered = sorted(
        shot_records.values(),
        key=lambda x: int(
            x["source_row"]
        ),
    )

    lines = [
        "-- Auto-generated caption inserts",
        "BEGIN;",
        "",
    ]

    for rec in ordered:
        lines.extend([
            (
                f"INSERT INTO "
                f"{CAPTION_TABLE} "
                f"(caption_id, shot_id, caption_text)"
            ),
            (
                "VALUES ("
                f"{sql_literal(rec['caption_id'])}, "
                f"{sql_literal(rec['shot_id'])}, "
                f"{sql_literal(rec['caption_text'])}"
                ")"
            ),
            (
                "ON CONFLICT (caption_id) "
                "DO UPDATE SET"
            ),
            "    shot_id = EXCLUDED.shot_id,",
            (
                "    caption_text = "
                "EXCLUDED.caption_text;"
            ),
            "",
        ])

    lines.append("COMMIT;")

    tmp_path = SQL_FILE.with_suffix(
        ".sql.tmp"
    )

    tmp_path.write_text(
        "\n".join(lines),
        encoding="utf-8",
    )

    tmp_path.replace(SQL_FILE)


def save_failures(failures):
    if failures:
        pd.DataFrame(
            failures
        ).to_csv(
            FAILED_FILE,
            index=False,
            encoding="utf-8",
        )

In [ ]:
# ============================================================
# CELL 7 - TEST NHANH 1 SHOT
# ============================================================
# Có thể bỏ qua cell này nếu đã test FPT API trước đó.
# Nếu shot đầu tiên dài, test có thể gọi tối đa 4 API request.

if not FPT_API_KEY:
    raise ValueError(
        "Hãy điền FPT_API_KEY ở CELL 1."
    )

if MODEL_NAME == "YOUR_VLM_MODEL_NAME":
    raise ValueError(
        "Hãy điền MODEL_NAME ở CELL 1."
    )

test_row = selected_df.iloc[0].to_dict()
test_video_id = str(
    test_row["video_id"]
)

test_video_path = None
test_images = []

try:
    _, test_video_path = download_video(
        test_video_id,
        video_url_map[test_video_id],
    )

    prepared = (
        make_missing_segment_sheets(
            test_row,
            test_video_path,
            completed_segment_keys=set(),
        )
    )

    segment_texts = []

    for item in prepared:
        test_images.append(
            item["image_path"]
        )

        display(
            Image.open(
                item["image_path"]
            )
        )

        text = caption_segment(item)

        segment_texts.append(
            (
                item["segment_index"],
                text,
            )
        )

        print(
            f"\nSEGMENT "
            f"{item['segment_index'] + 1}/"
            f"{item['total_segments']}:"
        )
        print(text)

    segment_texts.sort()

    print("\n=== FINAL TEST CAPTION ===")

    print(
        " ".join(
            text
            for _, text
            in segment_texts
        )
    )

finally:
    for path in test_images:
        remove_file(path)

    if test_video_path:
        remove_file(test_video_path)

In [ ]:
# ============================================================
# CELL 8 - CHẠY PIPELINE
# ============================================================

if not FPT_API_KEY:
    raise ValueError(
        "Hãy điền FPT_API_KEY ở CELL 1."
    )

if MODEL_NAME == "YOUR_VLM_MODEL_NAME":
    raise ValueError(
        "Hãy điền MODEL_NAME ở CELL 1."
    )


segment_records = (
    load_segment_checkpoint()
)

shot_records = (
    load_shot_checkpoint()
)

# Chỉ giữ checkpoint thuộc shard hiện tại.
selected_shot_ids = set(
    selected_df[
        "shot_id"
    ].astype(str)
)

segment_records = {
    key: rec
    for key, rec
    in segment_records.items()
    if key[0] in selected_shot_ids
}

shot_records = {
    shot_id: rec
    for shot_id, rec
    in shot_records.items()
    if shot_id in selected_shot_ids
}

pending_df = selected_df[
    ~selected_df[
        "shot_id"
    ].astype(str).isin(
        shot_records
    )
].copy()

print(
    f"Shot hoàn tất từ checkpoint: "
    f"{len(shot_records):,}"
)

print(
    f"Segment đã có checkpoint: "
    f"{len(segment_records):,}"
)

print(
    f"Shot còn cần xử lý: "
    f"{len(pending_df):,}"
)

failures = []
new_shot_count = 0

if len(pending_df) == 0:
    write_sql(shot_records)

    print(
        "Không còn shot cần xử lý."
    )

else:
    pending_video_ids = list(
        dict.fromkeys(
            pending_df[
                "video_id"
            ].tolist()
        )
    )

    for batch_start in range(
        0,
        len(pending_video_ids),
        VIDEO_BATCH_SIZE,
    ):
        batch_video_ids = (
            pending_video_ids[
                batch_start:
                batch_start
                + VIDEO_BATCH_SIZE
            ]
        )

        print(
            f"\n=== VIDEO BATCH "
            f"{batch_start // VIDEO_BATCH_SIZE + 1} | "
            f"{batch_video_ids} ==="
        )

        downloaded = {}
        batch_images = []

        try:
            # ----------------------------------------
            # A. DOWNLOAD VIDEO SONG SONG
            # ----------------------------------------
            with ThreadPoolExecutor(
                max_workers=min(
                    DOWNLOAD_WORKERS,
                    len(batch_video_ids),
                )
            ) as executor:
                future_map = {
                    executor.submit(
                        download_video,
                        video_id,
                        video_url_map[
                            video_id
                        ],
                    ): video_id
                    for video_id
                    in batch_video_ids
                }

                for future in tqdm(
                    as_completed(
                        future_map
                    ),
                    total=len(
                        future_map
                    ),
                    desc="Download videos",
                ):
                    video_id = (
                        future_map[future]
                    )

                    try:
                        _, video_path = (
                            future.result()
                        )

                        downloaded[
                            video_id
                        ] = video_path

                    except Exception as e:
                        failures.append({
                            "stage": "download",
                            "video_id": (
                                video_id
                            ),
                            "shot_id": "",
                            "segment_index": "",
                            "error": repr(e),
                        })

                        print(
                            f"[DOWNLOAD ERROR] "
                            f"{video_id}: {e}"
                        )

            if not downloaded:
                save_failures(
                    failures
                )
                continue

            batch_df = pending_df[
                pending_df[
                    "video_id"
                ].isin(
                    downloaded.keys()
                )
            ].copy()

            rows = [
                row.to_dict()
                for _, row
                in batch_df.iterrows()
            ]

            # ----------------------------------------
            # B/C. PROCESS THE SHOTS IN SMALL CHUNKS
            # ----------------------------------------
            for chunk_start in range(
                0,
                len(rows),
                SHOT_CHUNK_SIZE,
            ):
                chunk_rows = rows[
                    chunk_start:
                    chunk_start
                    + SHOT_CHUNK_SIZE
                ]

                prepared_segments = []

                completed_segment_keys = set(
                    segment_records.keys()
                )

                # ------------------------------------
                # B. CPU WORKERS -> CONTACT SHEETS
                # ------------------------------------
                with ThreadPoolExecutor(
                    max_workers=PROCESS_WORKERS
                ) as executor:
                    future_map = {}

                    for row in chunk_rows:
                        shot_id = str(
                            row["shot_id"]
                        )

                        # Nếu shot đã hoàn tất trong lúc chạy,
                        # không chuẩn bị lại.
                        if (
                            shot_id
                            in shot_records
                        ):
                            continue

                        video_id = str(
                            row["video_id"]
                        )

                        future = (
                            executor.submit(
                                make_missing_segment_sheets,
                                row,
                                downloaded[
                                    video_id
                                ],
                                completed_segment_keys,
                            )
                        )

                        future_map[
                            future
                        ] = row

                    for future in tqdm(
                        as_completed(
                            future_map
                        ),
                        total=len(
                            future_map
                        ),
                        desc="Build contact sheets",
                    ):
                        row = (
                            future_map[
                                future
                            ]
                        )

                        try:
                            items = (
                                future.result()
                            )

                            for item in items:
                                batch_images.append(
                                    item[
                                        "image_path"
                                    ]
                                )

                                prepared_segments.append(
                                    item
                                )

                        except Exception as e:
                            failures.append({
                                "stage": "process",
                                "video_id": (
                                    row[
                                        "video_id"
                                    ]
                                ),
                                "shot_id": (
                                    row[
                                        "shot_id"
                                    ]
                                ),
                                "segment_index": "",
                                "error": repr(e),
                            })

                            print(
                                f"[PROCESS ERROR] "
                                f"{row['shot_id']}: "
                                f"{e}"
                            )

                # ------------------------------------
                # C. API WORKERS
                # ------------------------------------
                if prepared_segments:
                    with ThreadPoolExecutor(
                        max_workers=API_WORKERS
                    ) as executor:
                        future_map = {
                            executor.submit(
                                caption_segment,
                                item,
                            ): item
                            for item
                            in prepared_segments
                        }

                        for future in tqdm(
                            as_completed(
                                future_map
                            ),
                            total=len(
                                future_map
                            ),
                            desc="FPT AI segments",
                        ):
                            item = (
                                future_map[
                                    future
                                ]
                            )

                            try:
                                segment_caption = (
                                    future.result()
                                )

                                record = {
                                    "shot_id": (
                                        item[
                                            "shot_id"
                                        ]
                                    ),
                                    "video_id": (
                                        item[
                                            "video_id"
                                        ]
                                    ),
                                    "shot_index": (
                                        item[
                                            "shot_index"
                                        ]
                                    ),
                                    "source_row": (
                                        item[
                                            "source_row"
                                        ]
                                    ),
                                    "segment_index": (
                                        item[
                                            "segment_index"
                                        ]
                                    ),
                                    "total_segments": (
                                        item[
                                            "total_segments"
                                        ]
                                    ),
                                    "segment_start_ms": (
                                        item[
                                            "segment_start_ms"
                                        ]
                                    ),
                                    "segment_end_ms": (
                                        item[
                                            "segment_end_ms"
                                        ]
                                    ),
                                    "segment_caption": (
                                        segment_caption
                                    ),
                                }

                                # CHECKPOINT NGAY
                                append_jsonl(
                                    SEGMENT_CHECKPOINT_FILE,
                                    record,
                                )

                                segment_records[
                                    (
                                        record[
                                            "shot_id"
                                        ],
                                        record[
                                            "segment_index"
                                        ],
                                    )
                                ] = record

                            except Exception as e:
                                failures.append({
                                    "stage": "api",
                                    "video_id": (
                                        item[
                                            "video_id"
                                        ]
                                    ),
                                    "shot_id": (
                                        item[
                                            "shot_id"
                                        ]
                                    ),
                                    "segment_index": (
                                        item[
                                            "segment_index"
                                        ]
                                    ),
                                    "error": repr(e),
                                })

                                print(
                                    f"[API ERROR] "
                                    f"{item['shot_id']} "
                                    f"seg "
                                    f"{item['segment_index']}: "
                                    f"{e}"
                                )

                            finally:
                                remove_file(
                                    item[
                                        "image_path"
                                    ]
                                )

                # ------------------------------------
                # D. BUILD FINAL SHOT CAPTIONS
                # ------------------------------------
                for row in chunk_rows:
                    shot_id = str(
                        row["shot_id"]
                    )

                    if shot_id in shot_records:
                        continue

                    final_record = (
                        build_final_shot_record(
                            row,
                            segment_records,
                        )
                    )

                    if final_record is None:
                        continue

                    # CHECKPOINT SHOT NGAY
                    append_jsonl(
                        SHOT_CHECKPOINT_FILE,
                        final_record,
                    )

                    shot_records[
                        shot_id
                    ] = final_record

                    new_shot_count += 1

                    if (
                        new_shot_count
                        % SQL_FLUSH_EVERY
                        == 0
                    ):
                        write_sql(
                            shot_records
                        )

                # Sau mỗi chunk vẫn flush SQL + failed.
                write_sql(
                    shot_records
                )

                save_failures(
                    failures
                )

        finally:
            # Xóa ảnh còn sót.
            for path in batch_images:
                remove_file(path)

            # Xóa video sau khi xử lý xong batch.
            for path in downloaded.values():
                remove_file(path)

            # Xóa file tải dở.
            for path in VIDEO_DIR.glob(
                "*.part"
            ):
                remove_file(path)

    # Final flush
    write_sql(
        shot_records
    )

    save_failures(
        failures
    )


print("\n==============================")
print("DONE / CURRENT STATE")
print("==============================")

print(
    f"Shot có caption hoàn chỉnh: "
    f"{len(shot_records):,}/"
    f"{len(selected_df):,}"
)

print(
    f"Segment checkpoint: "
    f"{len(segment_records):,}"
)

print("SQL:", SQL_FILE)
print(
    "Segment checkpoint:",
    SEGMENT_CHECKPOINT_FILE,
)
print(
    "Shot checkpoint:",
    SHOT_CHECKPOINT_FILE,
)

if failures:
    print(
        f"Failed: {FAILED_FILE} "
        f"({len(failures)} lỗi)"
    )
else:
    print("Failed: 0")

In [ ]:
# ============================================================
# CELL 9 - REBUILD SQL BẤT CỨ LÚC NÀO
# ============================================================
# Cell này hữu ích nếu runtime dừng trước khi pipeline chạy tới cuối.
# Chỉ cần checkpoint còn tồn tại trong /kaggle/working,
# cell sẽ dựng lại captions.sql từ shots_checkpoint.jsonl.

current_shot_records = (
    load_shot_checkpoint()
)

# Chỉ giữ shard đang chạy.
current_shot_records = {
    shot_id: rec
    for shot_id, rec
    in current_shot_records.items()
    if shot_id
    in selected_shot_ids
}

write_sql(
    current_shot_records
)

print(
    f"Rebuilt SQL từ "
    f"{len(current_shot_records):,} "
    f"caption hoàn chỉnh."
)

print("SQL:", SQL_FILE)

In [ ]:
# ============================================================
# CELL 10 - KIỂM TRA OUTPUT
# ============================================================

for p in sorted(
    OUTPUT_DIR.iterdir()
):
    if p.is_file():
        print(
            f"{p.name:32s} "
            f"{p.stat().st_size / 1024:.2f} KB"
        )

print(
    "\nFile chính để import PostgreSQL:"
)
print(SQL_FILE)

print(
    "\nKhông cần zip trên Kaggle."
)